# Credit Scoring + PSI/CSI Monitoring + Champion-Challenger

**Mục tiêu:** mô phỏng vòng đời 1 credit scoring model — train → quyết định champion-challenger → giám sát drift (PSI/CSI).

> 📋 Guide từng bước: `plans/260608-1544-project-03-credit-scoring-psi-csi-guide/`. Mỗi section dưới = 1 phase. Ô code `# TODO` là phần BẠN viết — đọc phase file tương ứng trước khi điền.

**Storytelling rule:** mỗi section mở đầu bằng *"làm gì / vì sao / kỳ vọng thấy gì"* và kết bằng *1 câu insight*. Không dump cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42  # reproducible mọi nơi
pd.set_option('display.max_columns', None)

## 1. Data & quirks  ·  *Phase 1*

Load dataset + **tự đi tìm cái bẫy** (đây là phần JD chấm "validate against ground truth"). Đừng nhận `describe()` ở face value.

Tự phát hiện: target imbalance · missing (Income, Dependents) · outlier (Revolving, DebtRatio) · **sentinel 96/98** trong 3 cột PastDue.

In [ ]:
# TODO (Bạn viết — Phase 1):
# 1. load data/cs-training.csv
# 2. EDA: shape, target mean (imbalance?), isna(), describe()
# 3. tìm sentinel 96/98 trong NumberOfTime30-59 / 60-89 / 90DaysLate
# 4. ghi lại soundbite quirks bạn tìm được


In [ ]:
# TODO (Bạn viết — Phase 1): giả lập 6 batch + inject drift
# - chia 6 batch đều nhau (shuffle seed 42)
# - chọn 1-2 feature, shift phân phối ở batch 4-6
# - LƯU ground-truth drift (dict: feature, batches, factor) để Phase 7 verify
# - split: batch 1-3 = train, batch 4-6 = test/monitor (KHÔNG random split)

DRIFT_GROUND_TRUTH = {}  # bạn điền


## 2. Data Prep & Feature Engineering  ·  *Phase 2*

Làm sạch quirks. **Fit transform trên TRAIN, apply lên TEST** (chống leakage — interview gotcha #1).

In [ ]:
# TODO (Bạn viết — Phase 2):
# - prep_fit(train) -> params (medians, caps);  prep_apply(df, params)
# - missing: median impute + flag;  outlier: winsorize p1/p99;  sentinel 96/98: xử lý riêng
# - 3-5 feature mới (total_past_due, has_been_late, ...), comment justify từng cái


## 3. WoE / IV  ·  *Phase 3*

Tự code WoE/IV (KHÔNG xài lib). IV lọc feature; **IV > 0.5 → nghi leakage**, không phải feature vàng. Fit bins trên train, apply test.

In [ ]:
# TODO (Bạn viết — Phase 3):
# - woe_iv(feature, target, bins): bin -> %good/%bad -> WoE=ln(%good/%bad) -> IV
# - smoothing tránh log(0); giữ NaN thành 1 bin riêng; kiểm tra dấu WoE
# - bảng IV mọi feature -> lọc -> flag IV>0.5 -> điều tra leakage
# - apply_woe(df, bin_map) cho LR input


## 4. Models — Champion (LR) vs Challenger (LightGBM)  ·  *Phase 4*

LR (WoE, champion — giải thích được) vs LightGBM (raw, challenger — mạnh hơn). 5-fold CV chỉ trên batch 1-3.

In [ ]:
# TODO (Bạn viết — Phase 4):
# - LR(class_weight='balanced', C=?) trên WoE features; in coef, kiểm tra dấu hợp lý
# - LightGBM(scale_pos_weight=?, 1 round tuning) trên raw features
# - 5-fold StratifiedKFold CV (AUC) trên batch 1-3; so mean±std


## 5. Evaluation — AUC/KS/Gini/Calibration/Lift  ·  *Phase 5*

Trên test (batch 4-6). KS/lift/calibration tự code logic. **Không báo accuracy** (imbalanced trap).

In [ ]:
# TODO (Bạn viết — Phase 5):
# - ks_statistic (max gap 2 CDF), gini = 2*AUC-1, lift_table by decile
# - calibration_curve trước/sau isotonic; quyết model nào cần calibrate
# - bảng so 2 model (train+test); đối chiếu target AUC>=0.80, KS>0.40


## 6. Champion-Challenger Decision  ·  *Phase 6*

Recall @ 5% FPR + bootstrap CI (paired) + decision rule **pre-registered**. Quy ra tiền. → readout.pdf.

In [ ]:
# TODO (Bạn viết — Phase 6):
# - recall_at_fpr(y, score, 0.05)
# - bootstrap_recall_diff (resample CÙNG index cho 2 model) -> mean + 95% CI
# - decision rule: ship nếu recall_lift>=3pp AND CI_lower>0 (chốt TRƯỚC khi xem số)
# - net_impact = loss_avoided - fp_cost (giả định nêu rõ)


## 7. PSI/CSI Monitoring  ·  *Phase 7*

Score 6 batch bằng champion. PSI (score) + CSI (feature). **Bin edges cố định từ baseline batch 1.** Verify bắt đúng drift đã inject ở Phase 1.

In [ ]:
# TODO (Bạn viết — Phase 7):
# - psi(base, curr, bins, edges): edges TỪ baseline, smoothing, PSI=sum((c-b)*ln(c/b))
# - PSI score trend batch 2->6 vs baseline; CSI heatmap feature x batch
# - VERIFY: CSI feature-đã-inject nhảy >0.25 ở batch 4-6; control giữ <0.1
# - drift table + recommend retrain


## 8. Limitations & Future work  ·  *Phase 8*

Time-window giả lập · fairness out-of-scope · WoE-in-CV (production).